# Optimización de Hiperparámetros

---

## Descripción

Este notebook aplica GridSearchCV con validación cruzada estratificada
para optimizar los hiperparámetros de los tres modelos de clasificación:
DecisionTreeClassifier, LogisticRegression y SVM.
Los mejores estimadores se comparan en una tabla resumen.

---

## Requisitos de Software

- pandas (>=1.1.0)
- numpy (>=2.0.0)
- scikit-learn (>=1.3)

In [78]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer,StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score, roc_auc_score

In [79]:
data = pd.read_csv('../data/dataset_clientes.csv')
data.head()

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,ultima_compra_dias,uso_app,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,356,Bajo,Estandar,3,1,Tienda,Lunes,22,3824,1
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,307,Medio,Premium,4,1,App,Martes,10,4148,0
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,232,Alto,Premium,4,1,App,Jueves,6,7200,0
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,165,Alto,Estandar,2,1,App,Domingo,16,1782,1
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,283,Bajo,Estandar,3,1,Web,Martes,8,3448,1


In [80]:
data = data.drop_duplicates()

# Preparación

In [81]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos via recorte por percentiles.
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X = X.astype('float64')
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)


def tratar_duplicados(X: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """
    Tratamiento de duplicados.
    Si drop=True elimina filas duplicadas, si no las deja.
    """
    return X.drop_duplicates() if drop else X


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values


class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

In [82]:
def evaluar(modelo, X_train, X_test, y_train, y_test):
    """
    Entrena el modelo y retorna métricas de clasificación.
    """
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }

In [83]:
features_num = [
    'edad', 'ingreso_mensual', 'gasto_mensual', 'deuda_total',
    'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos'
]
features_cat = [
    'genero', 'region', 'estado_civil', 'uso_app', 'tipo_plan', 'canal_registro'
]

numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

# SVM requiere escalar las features
numeric_transformer_svm = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_svm, features_num),
        ('cat', categorical_transformer,  features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

target_cls = 'abandono'

X_cls = data[features_num + features_cat]
y_cls = data[target_cls]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=29, stratify=y_cls
)

## DecisionTreeClassifier

In [84]:
pipeline_dtc = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeClassifier(random_state=29))
])

param_grid_dtc = {
    'modelo__max_depth':         [3, 5, 10],
    'modelo__min_samples_split': [2, 5, 10],
    'modelo__min_samples_leaf':  [1, 2, 4],
    'modelo__class_weight':      [None, 'balanced']
}

grid_dtc_tuned = GridSearchCV(
    pipeline_dtc, param_grid_dtc,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring='recall', n_jobs=-1
)
grid_dtc_tuned.fit(X_train_cls, y_train_cls)

print(f"Mejores parámetros: {grid_dtc_tuned.best_params_}")
print(f"Mejor F1 (CV)     : {grid_dtc_tuned.best_score_:.4f}")

Mejores parámetros: {'modelo__class_weight': 'balanced', 'modelo__max_depth': 5, 'modelo__min_samples_leaf': 1, 'modelo__min_samples_split': 2}
Mejor F1 (CV)     : 0.6325


In [85]:
metricas_dtc_tuned = evaluar(grid_dtc_tuned.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
print('DecisionTreeClassifier (tuned)')
print(f"  Accuracy : {metricas_dtc_tuned['accuracy']:.4f}")
print(f"  F1       : {metricas_dtc_tuned['f1']:.4f}")
print(f"  Precision: {metricas_dtc_tuned['precision']:.4f}")
print(f"  Recall   : {metricas_dtc_tuned['recall']:.4f}")
print(f"  ROC AUC  : {metricas_dtc_tuned['roc_auc']:.4f}")

DecisionTreeClassifier (tuned)
  Accuracy : 0.6098
  F1       : 0.5631
  Precision: 0.5065
  Recall   : 0.6339
  ROC AUC  : 0.6632


## Logistic Regression

In [86]:
pipeline_logreg = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LogisticRegression(class_weight='balanced', max_iter=10000, random_state=29))
])

param_grid_logreg = {
    'modelo__C':            [0.01, 0.1, 1, 10],
    'modelo__penalty':      ['l2'],
    'modelo__class_weight': [None, 'balanced']
}

grid_logreg_tuned = GridSearchCV(
    pipeline_logreg, param_grid_logreg,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring='recall', n_jobs=-1
)
grid_logreg_tuned.fit(X_train_cls, y_train_cls)

print(f"Mejores parámetros: {grid_logreg_tuned.best_params_}")
print(f"Mejor F1 (CV)     : {grid_logreg_tuned.best_score_:.4f}")

Mejores parámetros: {'modelo__C': 0.1, 'modelo__class_weight': 'balanced', 'modelo__penalty': 'l2'}
Mejor F1 (CV)     : 0.6311


In [87]:
metricas_logreg_tuned = evaluar(grid_logreg_tuned.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
print('LogisticRegression (tuned)')
print(f"  Accuracy : {metricas_logreg_tuned['accuracy']:.4f}")
print(f"  F1       : {metricas_logreg_tuned['f1']:.4f}")
print(f"  Precision: {metricas_logreg_tuned['precision']:.4f}")
print(f"  Recall   : {metricas_logreg_tuned['recall']:.4f}")
print(f"  ROC AUC  : {metricas_logreg_tuned['roc_auc']:.4f}")

LogisticRegression (tuned)
  Accuracy : 0.6190
  F1       : 0.5666
  Precision: 0.5163
  Recall   : 0.6276
  ROC AUC  : 0.6706


## SVM

In [88]:
pipeline_svm = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_svm),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        SVC(kernel='rbf', probability=True, random_state=29))
])

param_grid_svm = {
    'modelo__C':            [0.1, 1, 10],
    'modelo__kernel':       ['rbf', 'linear'],
    'modelo__class_weight': [None, 'balanced']
}

grid_svm_tuned = GridSearchCV(
    pipeline_svm, param_grid_svm,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring='recall', n_jobs=-1
)
grid_svm_tuned.fit(X_train_cls, y_train_cls)

print(f"Mejores parámetros: {grid_svm_tuned.best_params_}")
print(f"Mejor F1 (CV)     : {grid_svm_tuned.best_score_:.4f}")

Mejores parámetros: {'modelo__C': 0.1, 'modelo__class_weight': 'balanced', 'modelo__kernel': 'rbf'}
Mejor F1 (CV)     : 0.6435


In [89]:
metricas_svm_tuned = evaluar(grid_svm_tuned.best_estimator_, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
print('SVM (tuned)')
print(f"  Accuracy : {metricas_svm_tuned['accuracy']:.4f}")
print(f"  F1       : {metricas_svm_tuned['f1']:.4f}")
print(f"  Precision: {metricas_svm_tuned['precision']:.4f}")
print(f"  Recall   : {metricas_svm_tuned['recall']:.4f}")
print(f"  ROC AUC  : {metricas_svm_tuned['roc_auc']:.4f}")

SVM (tuned)
  Accuracy : 0.6218
  F1       : 0.5754
  Precision: 0.5187
  Recall   : 0.6459
  ROC AUC  : 0.6740


## Comparación final con modelos optimizados

In [ ]:
resumen = pd.DataFrame({
    'Modelo':    ['DecisionTreeClassifier', 'LogisticRegression', 'SVM'],
    'Accuracy':  [metricas_dtc_tuned['accuracy'],  metricas_logreg_tuned['accuracy'],  metricas_svm_tuned['accuracy']],
    'F1':        [metricas_dtc_tuned['f1'],         metricas_logreg_tuned['f1'],        metricas_svm_tuned['f1']],
    'Precision': [metricas_dtc_tuned['precision'],  metricas_logreg_tuned['precision'], metricas_svm_tuned['precision']],
    'Recall':    [metricas_dtc_tuned['recall'],     metricas_logreg_tuned['recall'],    metricas_svm_tuned['recall']],
    'ROC AUC':   [metricas_dtc_tuned['roc_auc'],    metricas_logreg_tuned['roc_auc'],   metricas_svm_tuned['roc_auc']]
})
resumen.set_index('Modelo', inplace=True)
print(resumen.to_string())

                        Accuracy        F1  Precision    Recall   ROC AUC
Modelo                                                                   
DecisionTreeClassifier   0.60975  0.563112   0.506546  0.633900  0.663241
LogisticRegression       0.61900  0.566553   0.516330  0.627599  0.670596
SVM                      0.62175  0.575358   0.518725  0.645873  0.674021


Exception ignored in: <function ResourceTracker.__del__ at 0x104795c60>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106addc60>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107a2dc60>
Traceback (most recent call last